# Online Evaluation Assignment — LangSmith

In this assignment you will run a complete **online evaluation** workflow on LangSmith:

1. Run an **example application** that is instrumented with tracing, so each call logs a **trace** (with nested **spans**) to a LangSmith project.
2. **Extract** those traces and spans back into this notebook.
3. **Evaluate** them with several LLM-as-judge metrics:
   - **whole-trace** metrics that score the end-to-end result, and
   - **span-level** metrics that score one specific step inside the trace.
4. **Log the evaluation results back to LangSmith** as feedback attached to the traces/spans.

> **Online evaluation** scores *real traces produced by a running application* — as opposed to *offline* evaluation, which scores a fixed curated dataset.

The example app is a tiny **retrieve → generate** question-answering pipeline. It produces a trace shaped like:

```
qa_pipeline                (root trace)
├── retrieve_context       (span: fetches supporting context)
└── generate_answer        (span: LLM produces the final answer)
```

We attach span-level metrics to `retrieve_context` and `generate_answer`, and whole-trace metrics to the root.


---
# Section 0 — Setup

### 0.1 — Install dependencies

In [ ]:
!pip install -q langsmith openai

### 0.2 — Imports

In [ ]:
import os
import json
import re
import time
from datetime import datetime, timedelta, timezone

from openai import AzureOpenAI
from langsmith import Client, traceable

### 0.3 — Configure Azure OpenAI

Fill in your Azure OpenAI details below.

> ⚠️ Replace the `YOUR_..._HERE` placeholders with your own values, and avoid sharing this notebook while real keys are pasted in.


In [ ]:
os.environ["OPENAI_API_TYPE"]    = "azure"
os.environ["OPENAI_API_BASE"]    = "https://YOUR-RESOURCE.openai.azure.com"   # <-- your endpoint
os.environ["OPENAI_API_VERSION"] = "2024-12-01-preview"
os.environ["OPENAI_API_KEY"]     = "YOUR_AZURE_OPENAI_API_KEY_HERE"           # <-- your key
os.environ["OPENAI_DEPLOYMENT"]  = "gpt-5"                                     # <-- your deployment name

azure = AzureOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    azure_endpoint=os.getenv("OPENAI_API_BASE"),
    api_version=os.getenv("OPENAI_API_VERSION"),
)
DEPLOYMENT = os.getenv("OPENAI_DEPLOYMENT")
print("Azure OpenAI client ready. Deployment:", DEPLOYMENT)

### 0.4 — Configure LangSmith + tracing

Paste your LangSmith key and pick a **project name**. Turning on `LANGSMITH_TRACING` is what makes the `@traceable` app in Section 1 log to this project.


In [ ]:
os.environ["LANGSMITH_API_KEY"] = "YOUR_LANGSMITH_API_KEY_HERE"   # <-- your key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "online-eval-demo"             # <-- traces land here

PROJECT_NAME = os.environ["LANGSMITH_PROJECT"]

client = Client(api_key=os.environ["LANGSMITH_API_KEY"])
print("LangSmith client initialized. Project:", PROJECT_NAME)

---
# Section 1 — The Example Application (instrumented)

A small retrieval-augmented QA pipeline. The `@traceable` decorator turns each function into a span; calling the root `qa_pipeline` produces one trace with two child spans.


### 1.1 — A tiny knowledge base + the retrieval span

`retrieve_context` is a `retriever` span. It does naive keyword-overlap retrieval over a small in-memory knowledge base and returns the best-matching passage.


In [ ]:
KNOWLEDGE_BASE = [
    "The capital of France is Paris, located on the Seine river.",
    "William Shakespeare wrote the tragedy Romeo and Juliet around 1595.",
    "At sea level (1 atm), water boils at 100 degrees Celsius.",
    "Jupiter is the largest planet in the solar system by mass and volume.",
    "During photosynthesis, plants primarily absorb carbon dioxide from the air.",
    "The first crewed Moon landing was Apollo 11 in 1969.",
    "The chemical symbol for gold is Au, from the Latin 'aurum'.",
    "Mount Everest is the highest mountain above sea level on Earth.",
]

@traceable(run_type="retriever", name="retrieve_context")
def retrieve_context(question: str) -> str:
    q_words = set(re.findall(r"[a-z]+", question.lower()))
    def overlap(passage):
        return len(q_words & set(re.findall(r"[a-z]+", passage.lower())))
    best = max(KNOWLEDGE_BASE, key=overlap)
    return best

### 1.2 — The generation span

`generate_answer` is an `llm` span. It answers the question **using only the retrieved context**, so we can later check groundedness at the span level.


In [ ]:
@traceable(run_type="llm", name="generate_answer")
def generate_answer(question: str, context: str) -> str:
    messages = [
        {"role": "system", "content": (
            "You are a question-answering assistant. Answer the user's question "
            "using ONLY the provided context. Be accurate and concise."
        )},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    resp = azure.chat.completions.create(
        model=DEPLOYMENT,
        messages=messages,
        max_completion_tokens=2000,
    )
    return resp.choices[0].message.content.strip()

### 1.3 — The root pipeline

`qa_pipeline` is the root `chain` span. It calls the two child spans and returns the final answer plus the context it used.


In [ ]:
@traceable(run_type="chain", name="qa_pipeline")
def qa_pipeline(question: str) -> dict:
    context = retrieve_context(question)
    answer = generate_answer(question, context)
    return {"answer": answer, "context": context}

---
# Section 2 — Generate Traces

Run the app over a few questions. Each call logs one trace to the ``online-eval-demo`` project.


In [ ]:
questions = [
    "What is the capital of France?",
    "Who wrote Romeo and Juliet?",
    "At what temperature does water boil at sea level?",
    "What is the largest planet in the solar system?",
    "What is the chemical symbol for gold?",
]

for q in questions:
    result = qa_pipeline(q)
    print(f"Q: {q}\nA: {result['answer']}\n")

# Tracing is sent in the background; give it a few seconds to flush to LangSmith.
time.sleep(5)
print("Done. Traces should now be visible in the LangSmith project.")

---
# Section 3 — Extract the Traces

Fetch recent runs from the project, group them by `trace_id`, and pull out the root run and the named child spans for each trace.


### 3.1 — Fetch recent runs

In [ ]:
LOOKBACK_HOURS = 1   # widen if your traces are older

now = datetime.now(timezone.utc)
start_time = now - timedelta(hours=LOOKBACK_HOURS)

runs = list(client.list_runs(project_name=PROJECT_NAME, start_time=start_time, end_time=now))
print(f"Fetched {len(runs)} runs from '{PROJECT_NAME}' in the last {LOOKBACK_HOURS} hour(s).")

if not runs:
    print("No runs found yet — tracing can lag a few seconds. Re-run Section 2, then this cell.")

### 3.2 — Group spans by trace and locate the root + named spans

Helpers to organise the flat list of runs into traces and to find a span by name.


In [ ]:
from collections import defaultdict

# Group all runs (spans) under their trace_id
trace_to_spans = defaultdict(list)
for r in runs:
    if r.trace_id:
        trace_to_spans[r.trace_id].append(r)

print(f"Found {len(trace_to_spans)} unique traces.\n")

def get_root(spans):
    """Root span has no parent."""
    for s in spans:
        if getattr(s, "parent_run_id", None) is None:
            return s
    return spans[0]

def get_span_by_name(spans, name):
    for s in spans:
        if s.name == name:
            return s
    return None

def as_text(value):
    """Best-effort flatten of a run inputs/outputs dict to a string."""
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    if isinstance(value, dict):
        for k in ("answer", "output", "content", "result", "text"):
            if k in value:
                v = value[k]
                return v if isinstance(v, str) else json.dumps(v, default=str)
        return json.dumps(value, default=str)
    return str(value)

# Quick peek at one trace
for tid, spans in list(trace_to_spans.items())[:1]:
    root = get_root(spans)
    print("Trace:", tid)
    print("  spans:", [s.name for s in spans])
    print("  question:", as_text(root.inputs))
    print("  final answer:", as_text(root.outputs))

---
# Section 4 — Define the Evaluators

Four LLM-as-judge metrics:

| Metric | Scope | What it checks |
|---|---|---|
| **Answer relevance** | whole trace | Does the final answer actually address the question? |
| **Answer helpfulness** | whole trace | Is the answer clear, complete, and useful? |
| **Retrieval relevance** | `retrieve_context` span | Is the retrieved passage relevant to the question? |
| **Groundedness** | `generate_answer` span | Is the answer supported by the retrieved context (no fabrication)? |

A shared helper parses the judge's JSON reply.


In [ ]:
def _parse_judge_json(text: str) -> dict:
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text or "", flags=re.DOTALL)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                pass
    return {}

def _judge(prompt: str) -> dict:
    resp = azure.chat.completions.create(
        model=DEPLOYMENT,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=2000,
    )
    return _parse_judge_json(resp.choices[0].message.content)

### 4.1 — Whole-trace metric: Answer relevance

In [ ]:
def eval_answer_relevance(question: str, answer: str) -> dict:
    prompt = f"""Judge whether the answer is relevant to the question.

Question:
{question}

Answer:
{answer}

Return ONLY JSON: {{"score": 1 or 0, "explanation": "<short reason>"}}"""
    parsed = _judge(prompt)
    return {"score": int(parsed.get("score", 0)), "comment": parsed.get("explanation", "")}

### 4.2 — Whole-trace metric: Answer helpfulness

In [ ]:
def eval_answer_helpfulness(question: str, answer: str) -> dict:
    prompt = f"""Rate how helpful the answer is for the question, considering clarity and completeness.

Question:
{question}

Answer:
{answer}

Return ONLY JSON: {{"score": <float 0.0-1.0>, "explanation": "<short reason>"}}"""
    parsed = _judge(prompt)
    try:
        score = float(parsed.get("score", 0))
    except Exception:
        score = 0.0
    return {"score": score, "comment": parsed.get("explanation", "")}

### 4.3 — Span metric: Retrieval relevance (on `retrieve_context`)

In [ ]:
def eval_retrieval_relevance(question: str, context: str) -> dict:
    prompt = f"""Judge whether the retrieved context is relevant and useful for answering the question.

Question:
{question}

Retrieved context:
{context}

Return ONLY JSON: {{"score": 1 or 0, "explanation": "<short reason>"}}"""
    parsed = _judge(prompt)
    return {"score": int(parsed.get("score", 0)), "comment": parsed.get("explanation", "")}

### 4.4 — Span metric: Groundedness (on `generate_answer`)

In [ ]:
def eval_groundedness(answer: str, context: str) -> dict:
    prompt = f"""Judge whether the answer is fully supported by the context and adds no unsupported claims.

Context:
{context}

Answer:
{answer}

Return ONLY JSON: {{"score": 1 or 0, "explanation": "<short reason>"}}"""
    parsed = _judge(prompt)
    return {"score": int(parsed.get("score", 0)), "comment": parsed.get("explanation", "")}

---
# Section 5 — Evaluate Traces and Log Feedback Back

For every trace we extracted, run the four metrics and write each result back to LangSmith with `client.create_feedback`:

- **whole-trace** scores are attached to the **root run**, and
- **span** scores are attached to the **specific span's run id**.


In [ ]:
logged = 0

for tid, spans in trace_to_spans.items():
    root = get_root(spans)
    retrieve_span = get_span_by_name(spans, "retrieve_context")
    generate_span = get_span_by_name(spans, "generate_answer")

    # Pull the text we need
    question = as_text(root.inputs)
    answer = ""
    context = ""
    if isinstance(root.outputs, dict):
        answer = root.outputs.get("answer", as_text(root.outputs))
        context = root.outputs.get("context", "")
    else:
        answer = as_text(root.outputs)
    if not context and retrieve_span is not None:
        context = as_text(retrieve_span.outputs)

    print(f"\nTrace {tid}\n  Q: {question}\n  A: {answer}")

    # --- Whole-trace metrics -> log on root run ---
    rel = eval_answer_relevance(question, answer)
    client.create_feedback(run_id=root.id, key="answer_relevance",
                           score=rel["score"], comment=rel["comment"])

    helpf = eval_answer_helpfulness(question, answer)
    client.create_feedback(run_id=root.id, key="answer_helpfulness",
                           score=helpf["score"], comment=helpf["comment"])

    print(f"  [trace] relevance={rel['score']}  helpfulness={helpf['score']}")

    # --- Span metric: retrieval relevance -> log on retrieve_context span ---
    if retrieve_span is not None:
        retr = eval_retrieval_relevance(question, as_text(retrieve_span.outputs))
        client.create_feedback(run_id=retrieve_span.id, key="retrieval_relevance",
                               score=retr["score"], comment=retr["comment"])
        print(f"  [span: retrieve_context] retrieval_relevance={retr['score']}")

    # --- Span metric: groundedness -> log on generate_answer span ---
    if generate_span is not None:
        grnd = eval_groundedness(as_text(generate_span.outputs), context)
        client.create_feedback(run_id=generate_span.id, key="groundedness",
                               score=grnd["score"], comment=grnd["comment"])
        print(f"  [span: generate_answer] groundedness={grnd['score']}")

    logged += 1

print(f"\n\nDone. Logged feedback for {logged} trace(s).")

### 5.1 — View the results

Open **LangSmith → your project (`online-eval-demo`) → Traces**. Each trace now shows **`answer_relevance`** and **`answer_helpfulness`** feedback at the trace level, and opening a trace reveals **`retrieval_relevance`** on the `retrieve_context` span and **`groundedness`** on the `generate_answer` span.

You have completed a full online evaluation: produced real traces from an instrumented app, extracted them, scored them with whole-trace and span-level metrics, and logged the results back to LangSmith. ✅
